# tp-05j レースのクラス・コース・競馬場ごとの専用化は効くか

**実行場所: ホスト PC（stg）。** 「クラス（新馬・未勝利・1〜3勝・オープン・重賞・障害）や開催コース（競馬場・芝ダ・距離）によって、予測モデルを最適化すべきではないか」を、次の順で確かめる。採否は valid 2022〜2024 だけで決める（test 2025 は使わない）。

1. **診断**（アンサンブル v2 の OOF から）: 区分ごとの LogLoss、一様予測に対する改善（skill）、較正の傾き。弱い区分・較正が歪んだ区分があるか。
2. **区分別の再較正**（LOYO）: 区分ごとの 1 次補正（Platt）。
3. **専用モデル**（LightGBM、調整後パラメータ、同じ特徴量セット）:
   - separate = 区分の学習行だけで学習 / weighted = 全学習行を使い、区分外の重みを 0.25 に下げる。
   - 区分の族: クラス（7）、コース = 芝ダ × 距離帯（9）、競馬場（10）、競馬場 × 芝ダ（約 20）。
   - 学習行が 3,000 未満の区分は全体モデルで埋める。
4. **混合と選択**: 専用と全体の確率平均（50:50）。区分ごとに「専用 or 全体」を、**より過去の年の成績だけで**選ぶ（2023・2024 年で評価）。

基準 = 同じ設定（seed 0）の全体モデル。採用基準 = p_win の LogLoss の差の 95% 信頼区間の上限 < 0 かつ連対・複勝が悪化しない（0.0002 以内）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import segment_models as sm, tp05_data, win_models as wm, zoo, ensemble as en
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_j"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP05J_THREADS", detect_compute(reserve_cores=6).threads))
WEIGHTED = os.environ.get("TP05J_WEIGHTED", "grade,course").split(",")
mlflow.set_experiment("keiba-tp-segment")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
SEGS = {
    "grade": sm.grade_bucket(df["race_class"], df["surface"]).to_numpy(),
    "course": np.asarray(sm.course_key(df["surface"], df["distance"])),
    "venue": df["venue_code"].astype(str).str.zfill(2).to_numpy(),
    "venue_surface": sm.venue_surface(df["venue_code"], df["surface"]).to_numpy(),
}
for k, v in SEGS.items():
    print(k, pd.Series(v).nunique(), "区分 / 行数", pd.Series(v).value_counts().to_dict() if pd.Series(v).nunique() <= 10 else "")
print(f"行 {len(X):,} / 特徴量 {X.shape[1]} / threads {THREADS}")

## 1. 診断（アンサンブル v2 の OOF）
skill = 一様予測（1/頭数）に対する p_win の LogLoss の改善率。較正の傾き = y を logit(p) に回帰した係数（1 なら較正済み、<1 なら確率が極端すぎる）。

In [ ]:
key = ["race_id", "horse_id"]
ens = pd.read_parquet(OOF_DIR / "ensemble_v2.parquet")
M = ens.merge(meta[key + ["finish_position"]], on=key).merge(pd.DataFrame({**{"race_id": df["race_id"], "horse_id": df["horse_id"]}, **{f"seg_{k}": v for k, v in SEGS.items()}, "field": df["field_size"]}), on=key)
y1 = (M["finish_position"] == 1).astype(float).to_numpy()
uni = 1.0 / M["field"].to_numpy()
rows = []
for fam in SEGS:
    for s, g in M.groupby(f"seg_{fam}"):
        yy = (g["finish_position"] == 1).astype(float).to_numpy()
        ll = wm.binary_logloss(g["p_win"].to_numpy(), yy)
        llu = wm.binary_logloss(1.0 / g["field"].to_numpy(), yy)
        z = en.logit(np.clip(g["p_win"].to_numpy(), 1e-6, 1 - 1e-6)).reshape(-1, 1)
        lr = LogisticRegression(C=1e6, max_iter=300).fit(z, yy)
        rows.append({"族": fam, "区分": s, "行数": len(g), "レース数": g["race_id"].nunique(), "logloss_win": ll, "一様": llu, "skill": 1 - ll / llu, "較正の傾き": float(lr.coef_[0][0]), "切片": float(lr.intercept_[0]),
                     "1位的中": float((g.assign(r=g.groupby("race_id")["p_win"].rank(ascending=False, method="first")).query("r==1")["finish_position"] == 1).mean())})
DIAG = pd.DataFrame(rows)
DIAG.to_csv(OUT / "diagnosis.csv", index=False)
for fam in SEGS:
    print(fam); display(DIAG[DIAG["族"] == fam].drop(columns="族").set_index("区分").round(4))

## 2. 区分別の再較正（LOYO）
各区分の logit に 1 次補正（Platt）を、他の年だけで当てはめて当該年に適用する。

In [ ]:
race_m, yrs_m = M["race_id"].to_numpy(), M["year"].to_numpy()
Ym = {"win": (M["finish_position"] == 1).astype(float).to_numpy(), "top2": (M["finish_position"] <= 2).astype(float).to_numpy(), "top3": (M["finish_position"] <= 3).astype(float).to_numpy()}
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
rows = []
for fam in SEGS:
    seg = M[f"seg_{fam}"].to_numpy()
    d = {"族": fam}
    for t in TOT:
        p0 = M[f"p_{t}"].to_numpy(); z = en.logit(np.clip(p0, 1e-6, 1 - 1e-6)); out = p0.copy()
        for s in np.unique(seg):
            for y in np.unique(yrs_m):
                tr_ = (seg == s) & (yrs_m != y); te_ = (seg == s) & (yrs_m == y)
                if tr_.sum() < 500 or te_.sum() == 0 or len(np.unique(Ym[t][tr_])) < 2:
                    continue
                out[te_] = LogisticRegression(C=1e4, max_iter=200).fit(z[tr_].reshape(-1, 1), Ym[t][tr_]).predict_proba(z[te_].reshape(-1, 1))[:, 1]
        p1 = wm.normalize_to_total(race_m, out, TOT[t])
        b = wm.paired_bootstrap_diff(wm.per_race_loss(race_m, p0, Ym[t]), wm.per_race_loss(race_m, p1, Ym[t]), n_boot=500)
        d[f"Δ{t}"], d[f"{t}_hi"] = b["diff"], b["hi"]
    d["採用"] = bool(d["win_hi"] < 0 and d["Δtop2"] <= 2e-4 and d["Δtop3"] <= 2e-4)
    rows.append(d)
RECAL = pd.DataFrame(rows).set_index("族")
display(RECAL.round(5)); RECAL.to_csv(OUT / "recalibration.csv")

## 3. 専用モデル
基準は全体モデル（同じ特徴量・パラメータ・seed 0）。専用モデルは区分ごとに学習し、学習行が足りない区分は全体モデルで埋める。

In [ ]:
t0 = time.time()
from src.pipeline.models import zoo_runner
res, GLOBAL = zoo_runner.run_oof("lgb_global_s0", "lgb_binary", X, meta, folds, [0], THREADS, params=dict(LP))
print(f"全体モデル(seed 0): win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)")
fin_map = meta.set_index(key)["finish_position"]


def evaluate(oof, ref=GLOBAL, name=""):
    m = ref.merge(oof, on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": name}
    for t, k in (("win", 1), ("top2", 2), ("top3", 3)):
        yy = (fin <= k).astype(float)
        bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t}_b"].to_numpy(), yy), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t}_v"].to_numpy(), yy), n_boot=500)
        out[f"{t}_base"], out[f"{t}_new"], out[f"Δ{t}"], out[f"{t}_hi"] = wm.binary_logloss(m[f"p_{t}_b"].to_numpy(), yy), wm.binary_logloss(m[f"p_{t}_v"].to_numpy(), yy), bb["diff"], bb["hi"]
    # 年ごとの win の差（一貫性）
    for yr, g in m.assign(fin=fin).groupby("year_b" if "year_b" in m.columns else "year"):
        yy = (g["fin"] == 1).astype(float).to_numpy()
        out[f"Δwin_{int(yr)}"] = wm.binary_logloss(g["p_win_v"].to_numpy(), yy) - wm.binary_logloss(g["p_win_b"].to_numpy(), yy)
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 2e-4 and out["Δtop3"] <= 2e-4)
    return out


SPEC, ROWS = {}, []
for fam in SEGS:
    t0 = time.time()
    oof, info = sm.oof_segmented(X, meta, folds, SEGS[fam], LP["lgb"], LP["n_rounds"], THREADS, mode="separate", base_oof=GLOBAL)
    SPEC[f"{fam}/separate"] = oof
    ROWS.append(evaluate(oof, name=f"{fam}/separate"))
    print(f"{fam}/separate Δwin {ROWS[-1]['Δwin']:+.5f} (hi {ROWS[-1]['win_hi']:+.5f}) ({time.time() - t0:.0f}s)", flush=True)
    gc.collect()

In [ ]:
for fam in WEIGHTED:
    if fam not in SEGS:
        continue
    t0 = time.time()
    oof, info = sm.oof_segmented(X, meta, folds, SEGS[fam], LP["lgb"], LP["n_rounds"], THREADS, mode="weighted", weight_other=0.25, base_oof=GLOBAL)
    SPEC[f"{fam}/weighted"] = oof
    ROWS.append(evaluate(oof, name=f"{fam}/weighted"))
    print(f"{fam}/weighted Δwin {ROWS[-1]['Δwin']:+.5f} (hi {ROWS[-1]['win_hi']:+.5f}) ({time.time() - t0:.0f}s)", flush=True)
    gc.collect()
# 全体と専用の平均（50:50）
for name, oof in list(SPEC.items()):
    m = GLOBAL.merge(oof, on=key, suffixes=("_b", "_v"))
    mix = m[key + ["year_b"]].rename(columns={"year_b": "year"}).copy()
    for t, tot in (("win", 1.0), ("top2", 2.0), ("top3", 3.0)):
        mix[f"p_{t}"] = wm.normalize_to_total(m["race_id"].to_numpy(), 0.5 * m[f"p_{t}_b"].to_numpy() + 0.5 * m[f"p_{t}_v"].to_numpy(), tot)
    SPEC[name + "+全体(50:50)"] = mix
    ROWS.append(evaluate(mix, name=name + "+全体(50:50)"))
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "Δwin_2022", "Δwin_2023", "Δwin_2024", "採用"]].round(5)); R.to_csv(OUT / "specialized_vs_global.csv")

## 4. 区分ごとに「専用 or 全体」を選ぶ（過去の年だけで選択）
fold y の区分 s について、y より前の valid 年での専用モデルと全体モデルの LogLoss を比べ、専用が良い区分だけ専用を使う（最初の年 2022 は全体）。評価は 2023・2024 年。

In [ ]:
def select_by_past(spec_oof, name):
    m = GLOBAL.merge(spec_oof, on=key, suffixes=("_b", "_v")).merge(pd.DataFrame({**{"race_id": df["race_id"], "horse_id": df["horse_id"]}, "seg": SEGS[name.split("/")[0]]}), on=key)
    m["year"] = m["year_b"]
    m["fin"] = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    yy = (m["fin"] == 1).astype(float).to_numpy()
    m["l_b"] = -(yy * np.log(np.clip(m["p_win_b"], 1e-12, 1)) + (1 - yy) * np.log(np.clip(1 - m["p_win_b"], 1e-12, 1)))
    m["l_v"] = -(yy * np.log(np.clip(m["p_win_v"], 1e-12, 1)) + (1 - yy) * np.log(np.clip(1 - m["p_win_v"], 1e-12, 1)))
    use = np.zeros(len(m), bool)
    for y in (2023, 2024):
        past = m[m["year"] < y].groupby("seg")[["l_b", "l_v"]].mean()
        good = set(past.index[past["l_v"] < past["l_b"]])
        use |= ((m["year"] == y) & m["seg"].isin(good)).to_numpy()
    sel = m[m["year"].isin([2023, 2024])].copy()
    u = use[sel.index]
    out = sel[key + ["year"]].copy()
    for t in ("win", "top2", "top3"):
        out[f"p_{t}"] = np.where(u, sel[f"p_{t}_v"], sel[f"p_{t}_b"])
    return out.reset_index(drop=True), float(u.mean())


rows = []
for name in [n for n in SPEC if n.endswith("separate") or n.endswith("weighted")]:
    sel, frac = select_by_past(SPEC[name], name)
    ref = GLOBAL[GLOBAL["year"].isin([2023, 2024])]
    r = evaluate(sel, ref=ref, name=name + " → 過去の年で選択")
    r["専用を使った行の割合"] = frac
    rows.append(r)
SEL = pd.DataFrame(rows).set_index("variant")
display(SEL[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "専用を使った行の割合", "採用"]].round(5)); SEL.to_csv(OUT / "select_by_past.csv")

In [ ]:
# 区分ごとの差（専用 − 全体の p_win LogLoss。負なら専用が良い）: 最も効いた設定について
fam_best = R[~R.index.str.contains("50:50")].sort_values("Δwin").index[0]
sp = SPEC[fam_best]
m = GLOBAL.merge(sp, on=key, suffixes=("_b", "_v")).merge(pd.DataFrame({"race_id": df["race_id"], "horse_id": df["horse_id"], "seg": SEGS[fam_best.split("/")[0]]}), on=key)
m["fin"] = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
rows = []
for s, g in m.groupby("seg"):
    yy = (g["fin"] == 1).astype(float).to_numpy()
    rows.append({"区分": s, "行数": len(g), "全体": wm.binary_logloss(g["p_win_b"].to_numpy(), yy), "専用": wm.binary_logloss(g["p_win_v"].to_numpy(), yy)})
BYSEG = pd.DataFrame(rows).set_index("区分"); BYSEG["差(専用−全体)"] = BYSEG["専用"] - BYSEG["全体"]
print("最良の設定:", fam_best); display(BYSEG.round(5)); BYSEG.to_csv(OUT / "best_by_segment.csv")
report = {"git_commit": GIT_COMMIT, "diagnosis_file": "diagnosis.csv", "recalibration": RECAL.round(6).reset_index().to_dict("records"), "specialized": R.round(6).reset_index().to_dict("records"),
          "select_by_past": SEL.round(6).reset_index().to_dict("records"), "best_setting": fam_best, "lgb_params": LP}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-05j-segment-specialization", tags={"notebook": "tp-05j", "git_commit": GIT_COMMIT}):
    for n, r in R.iterrows():
        mlflow.log_metrics({f"delta_win__{n}".replace("/", "_").replace(" ", "")[:240]: float(r["Δwin"])})
    for f_ in ("diagnosis.csv", "recalibration.csv", "specialized_vs_global.csv", "select_by_past.csv", "best_by_segment.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_j")
print("採用された設定:", list(R.index[R["採用"]]) + list(SEL.index[SEL["採用"]]))